In [1]:
import pandas as pd

In [3]:
DATA_PATH = "data_reliability.csv"
df = pd.read_csv(DATA_PATH)
df['event_date'] = pd.to_datetime(df['event_date'])

# Quick peek at the columns and row count
print(f"Total rows loaded: {len(df)}")
print("Columns in dataset:", df.columns.tolist())
print("\nSample rows:")
print(df.head(), "\n")

Total rows loaded: 978278
Columns in dataset: ['user_id', 'event_name', 'event_date', 'event_time', 'operating_system']

Sample rows:
              user_id        event_name event_date     event_time  \
0  0030b0b0ab0eab0ea3        app_launch 2024-07-07  1720359836954   
1  0034cccc1f3c1f3c5c     config_result 2024-07-07  1720312891288   
2  006370709f659f6506        app_launch 2024-07-07  1720376390676   
3  0090dadabf40bf403d  user_data_result 2024-07-07  1720372900213   
4  010fd2d2d71ad71a3f        app_launch 2024-07-07  1720340155738   

  operating_system  
0          android  
1          android  
2          android  
3          android  
4          android   



In [5]:
start_date = pd.to_datetime("2024-07-07")
end_date   = pd.to_datetime("2024-07-14")

mask = (df['event_date'] >= start_date) & (df['event_date'] <= end_date)
df_range = df.loc[mask].copy()

print(f"Rows between {start_date.date()} and {end_date.date()}: {len(df_range)}\n")


Rows between 2024-07-07 and 2024-07-14: 978278



In [7]:
daily_active = (
    df_range
    .groupby('event_date')['user_id']
    .nunique()
    .rename('active_users')
    .reset_index()
)
purchase_counts = (
    df_range
    .loc[df_range['event_name'] == 'purchase']
    .groupby('event_date')
    .size()
    .rename('purchase_count')
    .reset_index()
)
daily_stats = pd.merge(daily_active, purchase_counts, on='event_date', how='left').fillna(0)
daily_stats['purchase_count'] = daily_stats['purchase_count'].astype(int)

print("Daily Active Users and Purchase Counts (July 7 – 14, 2024):")
print(daily_stats, "\n")

Daily Active Users and Purchase Counts (July 7 – 14, 2024):
  event_date  active_users  purchase_count
0 2024-07-07         28368             484
1 2024-07-08         31831             563
2 2024-07-09         42368            1075
3 2024-07-10         50636             956
4 2024-07-11         51891             983
5 2024-07-12         50626             872
6 2024-07-13         38577             628
7 2024-07-14         32426             558 



In [9]:
daily_stats['purchaser_rate_pct'] = (
    daily_stats['purchase_count'] / daily_stats['active_users'] * 100
)

print("Daily Purchaser User Rate (%):")
print(daily_stats[['event_date', 'active_users', 'purchase_count', 'purchaser_rate_pct']], "\n")


Daily Purchaser User Rate (%):
  event_date  active_users  purchase_count  purchaser_rate_pct
0 2024-07-07         28368             484            1.706148
1 2024-07-08         31831             563            1.768716
2 2024-07-09         42368            1075            2.537292
3 2024-07-10         50636             956            1.887985
4 2024-07-11         51891             983            1.894355
5 2024-07-12         50626             872            1.722435
6 2024-07-13         38577             628            1.627913
7 2024-07-14         32426             558            1.720841 



In [11]:
# Filter out only the “purchase” rows for July 9, 2024
july9_mask = (df_range['event_date'] == pd.to_datetime("2024-07-09")) & \
             (df_range['event_name'] == 'purchase')
july9_purchases = df_range.loc[july9_mask].copy()

total_purchase_rows_j9 = len(july9_purchases)
unique_users_j9 = july9_purchases['user_id'].nunique()

print(f"July 9, 2024 Purchase Event Details:")
print(f"  Total purchase rows:       {total_purchase_rows_j9}")
print(f"  Unique purchasing users:   {unique_users_j9}")

# Count how many times each user_id appears among July 9 purchases
counts_per_user_j9 = july9_purchases.groupby('user_id').size().rename('count').reset_index()
dupe_users_j9 = counts_per_user_j9.loc[counts_per_user_j9['count'] > 1].copy()

print(f"  Users with >1 'purchase' row on July 9: {len(dupe_users_j9)}")
print("  Sample of users with duplicated 'purchase' rows on July 9:")
print(dupe_users_j9.head(), "\n")

July 9, 2024 Purchase Event Details:
  Total purchase rows:       1075
  Unique purchasing users:   796
  Users with >1 'purchase' row on July 9: 279
  Sample of users with duplicated 'purchase' rows on July 9:
               user_id  count
1   00383838BA-BBA-B66      2
6   02187C7CEA-4EA-43C      2
8   02489999D6-FD6-FE6      2
9   02FB9B9B34-934-96F      2
14  0455D2D220-220-22B      2 



In [13]:
dup_mask = july9_purchases.duplicated(
    subset=['user_id', 'event_name', 'event_date', 'event_time', 'operating_system'],
    keep='first'
)
duplicate_rows = july9_purchases.loc[dup_mask].copy()

# 7.2 How many rows are exact duplicates?
num_duplicate_rows = len(duplicate_rows)
print(f"Number of exact duplicated purchase rows on July 9: {num_duplicate_rows}")

# 7.3 Which operating systems contributed these duplicates?
os_counts = duplicate_rows['operating_system'].value_counts()
print("Operating System distribution among duplicated rows on July 9:")
print(os_counts, "\n")

Number of exact duplicated purchase rows on July 9: 279
Operating System distribution among duplicated rows on July 9:
operating_system
ios    279
Name: count, dtype: int64 



In [15]:
# 8. Recompute July 9 Rate After Deduplication
# -------------------------------
# 8.1 Deduplicate July 9 purchase rows by dropping exact duplicates
july9_de_duped = july9_purchases.drop_duplicates(
    subset=['user_id', 'event_name', 'event_date', 'event_time', 'operating_system'],
    keep='first'
)

# 8.2 Count of deduped purchase rows and unique purchasers
deduped_purchase_count_j9 = len(july9_de_duped)
deduped_unique_users_j9 = july9_de_duped['user_id'].nunique()

print("After removing exact duplicates on July 9:")
print(f"  Deduped purchase rows:     {deduped_purchase_count_j9}")
print(f"  Unique purchasing users:   {deduped_unique_users_j9}")

# 8.3 Active users on July 9 (from daily_stats)
active_on_j9 = int(daily_stats.loc[daily_stats['event_date'] == pd.to_datetime("2024-07-09"), 'active_users'])

# 8.4 Recomputed Purchaser User Rate (%)
recomputed_rate_j9 = deduped_purchase_count_j9 / active_on_j9 * 100
print(f"  Recomputed Purchaser Rate: {recomputed_rate_j9:.4f}%\n")

After removing exact duplicates on July 9:
  Deduped purchase rows:     796
  Unique purchasing users:   796
  Recomputed Purchaser Rate: 1.8788%



C:\Users\kemal\AppData\Local\Temp\ipykernel_22752\4045474031.py:18: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  active_on_j9 = int(daily_stats.loc[daily_stats['event_date'] == pd.to_datetime("2024-07-09"), 'active_users'])
